# Изучение основ SQL

Этот ноутбук поможет вам разобраться в базовых командах SQL с использованием SQLite. Мы создадим учебную базу данных магазина и попрактикуемся в написании запросов.

## Содержание
1. [Подготовка окружения](#Подготовка-окружения)
2. [Основы выборки (SELECT, FROM)](#1.-Основы-выборки)
3. [Фильтрация данных (WHERE)](#2.-Фильтрация-данных)
4. [Сортировка (ORDER BY)](#3.-Сортировка)
5. [Агрегатные функции](#4.-Агрегатные-функции)
6. [Группировка (GROUP BY)](#5.-Группировка)
7. [Объединение таблиц (JOIN)](#6.-Объединение-таблиц)
8. [Практические задачи](#Практические-задачи)

## Подготовка окружения
Сначала создадим базу данных и наполним её данными.

In [49]:
import sqlite3
import pandas as pd

# Создаем подключение к базе данных в памяти
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Создаем таблицы
cursor.executescript('''
CREATE TABLE Categories (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE Products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    price REAL,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES Categories(id)
);

CREATE TABLE Orders (
    id INTEGER PRIMARY KEY,
    product_id INTEGER,
    quantity INTEGER,
    order_date TEXT,
    FOREIGN KEY (product_id) REFERENCES Products(id)
);

INSERT INTO Categories (name) VALUES ('Электроника'), ('Одежда'), ('Книги');

INSERT INTO Products (name, price, category_id) VALUES 
    ('Смартфон', 50000, 1),
    ('Ноутбук', 80000, 1),
    ('Футболка', 1500, 2),
    ('Джинсы', 3500, 2),
    ('SQL для начинающих', 1200, 3),
    ('Python Guide', 2000, 3);

INSERT INTO Orders (product_id, quantity, order_date) VALUES 
    (1, 2, '2023-10-01'),
    (3, 5, '2023-10-02'),
    (2, 1, '2023-10-03'),
    (5, 3, '2023-10-04'),
    (1, 1, '2023-10-05');
''')

def run_query(query):
    return pd.read_sql_query(query, conn)

print("База данных успешно создана!")

База данных успешно создана!


### 1. Основы выборки
Команда `SELECT` используется для выбора столбцов, а `FROM` — для указания таблицы.

In [50]:
# Выбрать все колонки из таблицы Products
query = "SELECT * FROM Products"
run_query(query)

,id,name,price,category_id
0,1,Смартфон,50000.0,1
1,2,Ноутбук,80000.0,1
2,3,Футболка,1500.0,2
3,4,Джинсы,3500.0,2
4,5,SQL для начинающих,1200.0,3
5,6,Python Guide,2000.0,3


### 2. Фильтрация данных
Используйте `WHERE` для добавления условий.

In [51]:
# Выбрать товары дороже 2000
query = "SELECT name, price FROM Products WHERE price > 2000"
run_query(query)

,name,price
0,Смартфон,50000.0
1,Ноутбук,80000.0
2,Джинсы,3500.0


### 3. Сортировка
`ORDER BY` позволяет упорядочить данные (ASC — по возрастанию, DESC — по убыванию).

In [52]:
# Сортировка по цене (от дорогих к дешевым)
query = "SELECT * FROM Products ORDER BY price DESC"
run_query(query)

,id,name,price,category_id
0,2,Ноутбук,80000.0,1
1,1,Смартфон,50000.0,1
2,4,Джинсы,3500.0,2
3,6,Python Guide,2000.0,3
4,3,Футболка,1500.0,2
5,5,SQL для начинающих,1200.0,3


### 4. Агрегатные функции
Агрегатные функции выполняют вычисления на наборе значений и возвращают одно значение:
- **`COUNT(*)`** — подсчитывает количество строк в выборке.
- **`SUM(column)`** — вычисляет сумму всех значений в числовом столбце.
- **`AVG(column)`** — находит среднее арифметическое значений.
- **`MAX(column)`** — находит самое большое (максимальное) значение.
- **`MIN(column)`** — находит самое маленькое (минимальное) значение.

In [53]:
# Подсчет общего количества товаров и средней цены
query = "SELECT COUNT(*) as total_count, AVG(price) as avg_price FROM Products"
run_query(query)

,total_count,avg_price
0,6,23033.333333


### 5. Группировка
`GROUP BY` объединяет строки с одинаковыми значениями в группы.

In [54]:
# Количество товаров в каждой категории
query = "SELECT category_id, COUNT(*) FROM Products GROUP BY category_id"
run_query(query)

,category_id,COUNT(*)
0,1,2
1,2,2
2,3,2


### 6. Объединение таблиц
`JOIN` позволяет объединять данные из разных таблиц по общему полю.

In [55]:
# Вывести название товара и название его категории
query = """
SELECT p.name as product_name, c.name as category_name 
FROM Products p 
JOIN Categories c ON p.category_id = c.id
"""
run_query(query)

,product_name,category_name
0,Смартфон,Электроника
1,Ноутбук,Электроника
2,Футболка,Одежда
3,Джинсы,Одежда
4,SQL для начинающих,Книги
5,Python Guide,Книги


## Описание данных

Перед решением задач ознакомьтесь со структурой таблиц:

1. **`Categories`** (Категории товаров):
    - `id` — уникальный идентификатор категории.
    - `name` — название категории ('Электроника', 'Одежда', 'Книги').

2. **`Products`** (Товары):
    - `id` — уникальный идентификатор товара.
    - `name` — название товара.
    - `price` — цена товара.
    - `category_id` — ID категории (связь с таблицей `Categories`).

3. **`Orders`** (Заказы):
    - `id` — уникальный идентификатор заказа.
    - `product_id` — ID купленного товара (связь с таблицей `Products`).
    - `quantity` — количество купленных единиц.
    - `order_date` — дата совершения заказа.

## Практические задачи

Напишите SQL-запросы для решения следующих задач:

### Задача 1
Выберите все заказы (`Orders`), где количество (`quantity`) больше 2.

In [56]:
# Ваше решение задачи 1
query = "SELECT * FROM Orders WHERE quantity > 2"
run_query(query)

,id,product_id,quantity,order_date
0,2,3,5,2023-10-02
1,4,5,3,2023-10-04


### Задача 2
Найдите самый дорогой товар в категории 'Электроника' (category_id = 1).

In [57]:
# Ваше решение задачи 2
query = "SELECT MAX(price),* FROM Products WHERE category_id = 1"
run_query(query)

,MAX(price),id,name,price,category_id
0,80000.0,2,Ноутбук,80000.0,1


### Задача 3
Посчитайте суммарную выручку (quantity * price) по всем заказам (потребуется JOIN между `Orders` и `Products`).

In [58]:
# Ваше решение задачи 3
query = """
SELECT SUM(o.quantity * p.price) AS total_sum
FROM Orders o
JOIN Products p ON o.product_id = p.id;
"""
run_query(query)

,total_sum
0,241100.0


### Задача 4
Выведите список категорий, в которых более 1 товара.

In [59]:
# Ваше решение задачи 4
query = """
SELECT c.name, COUNT(p.id) AS product_count
FROM Categories c
JOIN Products p ON c.id = p.category_id
GROUP BY c.name
HAVING COUNT(p.id) > 1;
"""
run_query(query)

,name,product_count
0,Книги,2
1,Одежда,2
2,Электроника,2
